## CalculateEmbeddings

This script:
- Loads list of sample points, each with 4 associated image files from an H5 data store
- Calculates an image embedding for each image file
- Saves the embeddings with the other metadata in the H5 store.

In [1]:
import io
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
import clip
import h5py

In [2]:
from directory_filepaths import *

print("Directory paths are:")
print("data_dir: ", data_dir)
print("lsoas_file: ", lsoas_file)
print("imd_file: ", imd_file)
print("h5_filename: ", h5_filename)



Directory paths are:
data_dir:  ../data/processed/
lsoas_file:  ../data/raw/spatial/LSOAs_2021/LSOA_2021_EW_BSC_V4.shp
imd_file:  ../data/raw/imd/File_2_-_IoD2025_Domains_of_Deprivation.xlsx
h5_filename:  ../data/processed/street_data.h5


### Load list of sample points

This contains points sampled along the road network in 1-SampleStreetNetwork.ipynb  

Each point has an ID, a latitude, a longitude, and 4 image files associated with it (these are sampled in each of the 4 cardinal directions from the sample point)  

This script will also create an 'embeddings' slot that it will fill with a list of embeddings for each of the 4 images

In [3]:
with h5py.File(h5_filename, "r") as f:
    print(f"H5 file contains {len(f["point_id"])} items")
    print("Keys are: ", list(f.keys()))

#points_data_cache = data_dir + "sample_points_cache/points_data_cache.pkl"
#with open(points_data_cache, "rb") as f:
#        point_records = pickle.load(f)
#print(f"Cache currently has {len(point_records)} points.")

H5 file contains 18897 items
Keys are:  ['date', 'embeddings_clip', 'image_paths', 'images_jpeg', 'images_present', 'latitude', 'longitude', 'point_id']


# Compute the Embeddings

In [4]:
# Define model and device to run it

def get_device():
    if torch.backends.mps.is_available():  # macs
        return torch.device("mps")
    elif torch.cuda.is_available():  # for completeness if you ever run on CUDA
        return torch.device("cuda")
    else:
        return torch.device("cpu")

device = get_device()
print("Using device:", device)
model, preprocess = clip.load("ViT-B/32", device=device)

Using device: mps


## Create embedding for each image and find similarity to categories 
- Create embedding for image
- Find similarity score to text embedding for each category
- Convert similarity score to a "probability-like number" using softmax

Prepare the database (we need an embeddings column)

In [5]:
dim = 512  # Length of CLIP embeddings
overwrite_embeddings = False  # Set to True to recalculate and overwrite existing embeddings
calculate_embeddings = True  # Will be set to False if we skip (i.e. if embeddins have already been calculated)

with h5py.File(h5_filename, "a") as f:
    N = f["point_id"].shape[0]

    if "embeddings_clip" in f:
        if not overwrite_embeddings:
            existing = f["embeddings_clip"][:]
            n_populated = np.count_nonzero(~np.all(np.isnan(existing), axis=-1)) // 4
            print(f"embeddings_clip already exists with {n_populated}/{N} points populated.")
            print("Skipping. Set overwrite_embeddings = True to recalculate.")
            calculate_embeddings = False
        else:
            print("overwrite_embeddings is True, replacing existing embeddings.")
            del f["embeddings_clip"]
            f.create_dataset(
                "embeddings_clip",
                shape=(N, 4, dim),
                dtype="float32",
                fillvalue=np.nan
            )
            print(f"Created embeddings_clip dataset with shape ({N}, 4, {dim})")
    else:
        f.create_dataset(
            "embeddings_clip",
            shape=(N, 4, dim),
            dtype="float32",
            fillvalue=np.nan
        )
        print(f"Created embeddings_clip dataset with shape ({N}, 4, {dim})")

embeddings_clip already exists with 18896/18897 points populated.
Skipping. Set overwrite_embeddings = True to recalculate.


Calculate the embeddings (unless they have already been calculated)

In [6]:
def load_pil_from_h5(f, row_idx, slot):
    """
    Returns a PIL.Image for the given row and slot, reading 'images_jpeg' bytes.
    Raises FileNotFoundError if images_present is False.
    """
    if not bool(f["images_present"][row_idx, slot]):
        raise FileNotFoundError(f"No image stored at row {row_idx}, slot {slot}")

    jpeg_bytes = f["images_jpeg"][row_idx, slot].tobytes()
    return Image.open(io.BytesIO(jpeg_bytes)).convert("RGB")


def embed_clip_pil(pil_image):
    """
    Compute a CLIP embedding from a PIL image.
    Returns a (D,) float32 numpy array (unit-normalized).
    """
    image_tensor = preprocess(pil_image).unsqueeze(0).to(device)

    with torch.no_grad():
        raw = model.encode_image(image_tensor)
        emb = raw / raw.norm(dim=-1, keepdim=True)

    return emb.squeeze(0).detach().cpu().numpy().astype("float32")


if not calculate_embeddings:
    print("Skipping embedding calculation (embeddings already exist).")
else:
    with h5py.File(h5_filename, "a") as f:
        if "embeddings_clip" not in f:
            raise KeyError("No embeddings file, this should have been created above.")

        emb_ds = f["embeddings_clip"]    # shape: (N, 4, D)
        N, _, D = emb_ds.shape
        print(f"Embeddings shape: {emb_ds.shape}")

        for i in tqdm(range(N), desc="Embedding CLIP from HDF5", unit="point"):
            for j in range(4):
                try:
                    # Decode image from HDF5
                    pil_img = load_pil_from_h5(f, i, j)

                    # Compute embedding and write it
                    emb = embed_clip_pil(pil_img)  # (D,)
                    if emb.shape[0] != D:
                        raise ValueError(f"Embedding dim mismatch: got {emb.shape[0]}, expected {D}")
                    emb_ds[i, j, :] = emb
                except FileNotFoundError:
                    # Missing image -> leave NaNs
                    tqdm.write(f"Missing image at {i}, {j}, not calculating the embedding")
                    emb_ds[i, j, :] = np.nan
                except Exception as e:
                    # Any other error -> write NaNs to mark as missing/bad
                    emb_ds[i, j, :] = np.nan
                    tqdm.write(f"Error at row {i}, slot {j}: {e}")

    print("Finished")

Skipping embedding calculation (embeddings already exist).


Done!

To finish just print a bit of information about what's in the H5 data store

In [7]:
with h5py.File(h5_filename, "r") as f:
    print(f"H5 data store: {h5_filename}")
    print(f"{'Dataset':<20} {'Shape':<25} {'Dtype'}")
    print("-" * 60)
    for key in sorted(f.keys()):
        ds = f[key]
        print(f"{key:<20} {str(ds.shape):<25} {ds.dtype}")

    N = f["point_id"].shape[0]
    n_images = np.count_nonzero(f["images_present"][:])
    print(f"\nTotal points: {N}")
    print(f"Images present: {n_images} / {N * 4}")

    if "embeddings_clip" in f:
        emb = f["embeddings_clip"][:]
        n_emb = np.count_nonzero(~np.all(np.isnan(emb), axis=-1))
        print(f"Embeddings populated: {n_emb} / {N * 4}")

H5 data store: ../data/processed/street_data.h5
Dataset              Shape                     Dtype
------------------------------------------------------------
date                 (18897,)                  |S10
embeddings_clip      (18897, 4, 512)           float32
image_paths          (18897, 4)                |S512
images_jpeg          (18897, 4)                object
images_present       (18897, 4)                bool
latitude             (18897,)                  float64
longitude            (18897,)                  float64
point_id             (18897,)                  int32

Total points: 18897
Images present: 75586 / 75588
Embeddings populated: 75586 / 75588
